<a href="https://colab.research.google.com/github/Abi05-04/Neural-Network/blob/main/Minimal_Seq2Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Dummy data:
input_seq = torch.tensor([[1, 2, 3]], dtype=torch.long)   # (batch, seq_len)
target_seq = torch.tensor([[3, 2, 1]], dtype=torch.long)

vocab_size = 10 # toy vocabulary size
embedding_dim = 8
hidden_size = 16

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)         # (batch, seq_len, embed_dim)
        # The nn.RNN expects input of shape (batch, seq_len, input_size), here input_size is embedding_dim
        # If no initial hidden state is provided, it defaults to zeros.
        outputs, hidden = self.rnn(embedded)
        return hidden

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, target, hidden):
        embedded = self.embedding(target)      # (batch, seq_len, embed_dim)
        # The nn.RNN expects input of shape (batch, seq_len, input_size) and an optional hidden state (h_0).
        # Here, 'embedded' is the input, and 'hidden' is the initial hidden state (e.g., from the encoder).
        outputs, _ = self.rnn(embedded, hidden)# outputs: (batch, seq_len, hidden), hidden: (1, batch, hidden)
        logits = self.fc(outputs)              # (batch, seq_len, vocab_size)
        return logits

In [ ]:
encoder = Encoder(vocab_size, embedding_dim, hidden_size)
decoder = Decoder(vocab_size, embedding_dim, hidden_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)

In [ ]:
encoder_hidden = encoder(input_seq)  # hidden: (1, batch, hidden)
logits = decoder(target_seq, encoder_hidden)

In [ ]:
logits.shape # (batch, seq_len, vocab_size)

torch.Size([1, 3, 10])

In [ ]:
# One epoch training
encoder_hidden = encoder(input_seq)  # hidden: (1, batch, hidden)
logits = decoder(target_seq, encoder_hidden)

# **During Inference**

In [ ]:
def inference(input_seq):
    with torch.no_grad():
        hidden = encoder(input_seq)

        # Start with  token; here we use 0 as dummy
        decoder_input = torch.tensor([[0]])   # (batch=1, seq_len=1)
        outputs = []

        for _ in range(input_seq.size(1)):
            logits = decoder(decoder_input, hidden)
            pred_token = logits.argmax(2)     # (batch, 1)
            outputs.append(pred_token.item())

            decoder_input = pred_token        # Next input is previous output

        return outputs


In [ ]:
print("Predicted:", inference(input_seq))  # Should roughly be [3, 2, 1]

Predicted: [6, 6, 6]


# **In case of Teacher Forcing**

In [ ]:
class EncoderTF(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)         # (batch, seq_len, embed_dim)
        outputs, hidden = self.rnn(embedded) # outputs: (batch, seq_len, hidden), hidden: (1, batch, hidden)
        return hidden

In [ ]:
class DecoderTF(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, target_seq, hidden, teacher_forcing_ratio=0.5):
        batch_size, target_len = target_seq.size()
        outputs = []

        input_token = target_seq[:, 0].unsqueeze(1)  # first input ( assumed)

        for t in range(1, target_len):
            embedded = self.embedding(input_token)               # (batch, 1, embed_dim)
            output, hidden = self.rnn(embedded, hidden)          # (batch, 1, hidden)
            logits = self.fc(output.squeeze(1))                  # (batch, vocab)
            outputs.append(logits.unsqueeze(1))                  # collect for all time steps

            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            next_input = target_seq[:, t] if teacher_force else logits.argmax(1)
            input_token = next_input.unsqueeze(1)

        return torch.cat(outputs, dim=1)  # (batch, target_len-1, vocab)

In [ ]:
encoder2 = EncoderTF(vocab_size, embedding_dim, hidden_size)
decoder2 = DecoderTF(vocab_size, embedding_dim, hidden_size)

criterion2 = nn.CrossEntropyLoss()
optimizer2 = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)

In [ ]:
# Forward pass
encoder_hidden2 = encoder2(input_seq)  # (1, batch, hidden)
logits2 = decoder2(target_seq, encoder_hidden2, teacher_forcing_ratio=0.7)

# Compute loss (note we shifted target by 1)
loss2 = criterion2(logits2.view(-1, vocab_size), target_seq[:, 1:].reshape(-1))
loss2.backward()
optimizer2.step()

print("Training loss with teacher forcing:", loss2.item())


Training loss with teacher forcing: 2.348672866821289


In [ ]:
def inference_tf(input_seq, max_len=5):
    with torch.no_grad():
        hidden = encoder2(input_seq)
        input_token = torch.tensor([[0]])  # start with  (assumed 0)
        outputs = []

        for _ in range(max_len):
            embedded = decoder2.embedding(input_token)
            output, hidden = decoder2.rnn(embedded, hidden)
            logits = decoder2.fc(output.squeeze(1))
            pred_token = logits.argmax(1)
            outputs.append(pred_token.item())
            input_token = pred_token.unsqueeze(1)

        return outputs

print("Inference output:", inference_tf(input_seq))

Inference output: [0, 0, 0, 0, 0]
